In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from itertools import combinations
from matplotlib.patches import Rectangle
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score



In [ ]:
targeted_error = np.array([1, 1, 1, 5, 5, 5, 10, 10, 10])  # for 9 properties
# === 1. Load Data ===
df_full = pd.read_csv('../../../data/training/data_till_now.csv')

prop_dict = {}
    
for i, prop in enumerate(['D_11', 'D_11H', 'D_14', 'SE_T11', 'SE_T11H', 'SE_T14', 'BM_T11', 'BM_T11H', 'BM_T14']):
    prop_dict[f'Property {i+1}'] = prop

mean_cols = ['D_11', 'D_11H', 'D_14', 'SE_T11', 'SE_T11H', 'SE_T14', 'BM_T11', 'BM_T11H', 'BM_T14']
error_cols = [f'error_{i}' for i in mean_cols]


y_error_all = df_full[error_cols].values # y_error_all = np.abs(100 * (y - target_y) / target_y)



In [ ]:
# Setup
property_indices = list(range(9))
prop_pairs = list(combinations(property_indices, 2))  # 9C2 = 36
fig, axs = plt.subplots(6, 6, figsize=(15, 14))
axs = axs.flatten()

for idx, (i, j) in enumerate(prop_pairs):
    ax = axs[idx]
    
    # Data for the current property pair
    x = y_error_all[:, i].reshape(-1, 1)
    y = y_error_all[:, j]

    # Scatter plot
    ax.scatter(x, y, s=10, alpha=0.7, label='Data')
    
    # Linear regression
    model = LinearRegression()
    model.fit(x, y)
    y_pred = model.predict(x)
    r2 = r2_score(y, y_pred)
    slope = model.coef_[0]
    intercept = model.intercept_
    
    # Plot the linear fit
    x_vals = np.linspace(x.min(), x.max(), 100).reshape(-1, 1)
    y_vals = model.predict(x_vals)
    ax.plot(x_vals, y_vals, color='red', linewidth=1, label='Linear Fit')

    # Axis labels
    ax.set_xlabel(prop_dict[f"Property {i+1}"], fontsize=8)
    ax.set_ylabel(prop_dict[f"Property {j+1}"], fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
   
    ax.tick_params(labelsize=7)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)

    # Add red box for target error zone
    rect = Rectangle(
        (0, 0),  # bottom left corner
        targeted_error[i],  # width
        targeted_error[j],  # height
        linewidth=1,
        edgecolor='red',
        facecolor='none'
    )
    ax.add_patch(rect)

    # Annotation with equation and R²
    ax.text(0.05, 0.95,
            f'y = {slope:.2f}x + {intercept:.2f}\nR² = {r2:.2f}',
            transform=ax.transAxes,
            fontsize=6,
            verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

plt.tight_layout()    
plt.show()


In [ ]:
# === 2. Compute Manhattan Distance ===
manhattan_distances = np.sum(np.abs(y_error_all), axis=1)

# Check if a point is inside the red-box (i.e., within all target error thresholds)
# within_target_zone = np.all(y_error_all <= targeted_error, axis=1)
within_target_zone = manhattan_distances <= np.sum(targeted_error)

# === 3. Plot Manhattan Distance ===
plt.figure(figsize=(10, 5))
plt.hist(manhattan_distances, bins=100, alpha=0.7, label='All Sets')
plt.hist(manhattan_distances[within_target_zone], bins=50, alpha=0.7, label='Within Target Zone')
plt.axvline(x=np.sum(targeted_error), color='red', linestyle='--', label='Sum of Target Thresholds\n (3*1+ 3*5 + 3*10 = 48)')
plt.xlabel("Manhattan Distance")
plt.ylabel("Count")
plt.title("Distribution of Manhattan Distances (L1 Norm) from Experimental values in 9D Error Space")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()



In [ ]:
# within_target_zone = manhattan_distances <= 55
# === 5. Identify and Save Points Within Target Zone ===
good_indices = np.where(within_target_zone)[0]
df_within_target = df_full.loc[within_target_zone].copy()
df_within_target["manhattan_distance"] = manhattan_distances[within_target_zone]

print(f"Total parameter sets within target zone: {len(df_within_target)}")
print("Indices of best sets:", good_indices.tolist())

# Optional: sort by Manhattan distance (ascending)
df_within_target_sorted = df_within_target.sort_values("manhattan_distance")

# Save to CSV
df_within_target_sorted.to_csv(f"../../../data/training/optimal_parameter_sets_{len(df_within_target)}.csv", index=False)


In [ ]:
# === 3x3 Manhattan Distance (Single Property) Plots ===
fig, axs = plt.subplots(3, 3, figsize=(14, 10))
axs = axs.flatten()

for idx, ax in enumerate(axs):
    prop_name = mean_cols[idx]
    prop_errors = y_error_all[:, idx]
    prop_thresh = targeted_error[idx]

    ax.hist(prop_errors, bins=60, alpha=0.7, label='All Sets')
    # ax.hist(prop_errors[prop_errors <= prop_thresh], bins=40, alpha=0.7, label='Within Threshold')

    ax.axvline(prop_thresh, color='red', linestyle='--', label=f'Threshold = {prop_thresh}')

    ax.set_title(f"{prop_name} Error")
    ax.set_xlabel("Absolute % Error")
    ax.set_ylabel("Count")
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()